In [0]:
from pyspark.sql import functions as F

# 1. Read Silver Fact tables
df_shift = spark.table("group_project.bronze.factshift")
df_work = spark.table("group_project.bronze.factwork")
df_shrinkage = spark.table("group_project.bronze.factshrinkage")

# 2. Find earliest and latest dates across all three tables
date_ranges = [
    df_shift.select(
        F.to_date("scheduled_start").alias("start_date"),
        F.to_date("scheduled_end").alias("end_date")
    ),
    df_work.select(
        F.to_date("scheduled_start").alias("start_date"),
        F.to_date("scheduled_end").alias("end_date")
    ),
    df_shrinkage.select(
        F.to_date("start_datetime").alias("start_date"),
        F.to_date("end_datetime").alias("end_date")
    )
]

all_dates = date_ranges[0].unionByName(date_ranges[1]).unionByName(date_ranges[2])

date_limits = all_dates.select(
    F.min("start_date").alias("min_date"),
    F.max("end_date").alias("max_date")
).first()

start_date = date_limits["min_date"]
end_date = date_limits["max_date"]

print("DimDate starts:", start_date)
print("DimDate ends:", end_date)

# 3. Create one row for every calendar date
df_dim_date = spark.sql(f"""
    SELECT explode(
        sequence(
            to_date('{start_date}'),
            to_date('{end_date}'),
            interval 1 day
        )
    ) AS full_date
""")

# 4. Add calendar attributes
df_dim_date = (
    df_dim_date
    .withColumn("date_key", F.date_format("full_date", "yyyyMMdd").cast("int"))
    .withColumn("day", F.dayofmonth("full_date"))
    .withColumn("day_name", F.date_format("full_date", "EEEE"))
    .withColumn("week_number", F.weekofyear("full_date"))
    .withColumn("month_number", F.month("full_date"))
    .withColumn("month_name", F.date_format("full_date", "MMMM"))
    .withColumn("quarter", F.concat(F.lit("Q"), F.quarter("full_date")))
    .withColumn("year", F.year("full_date"))
    .withColumn("is_weekend", F.dayofweek("full_date").isin([1, 7]))
)

# 5. Display and validate
display(df_dim_date)
print("Total dates:", df_dim_date.count())

# 6. Create Gold schema if it does not exist
spark.sql("CREATE SCHEMA IF NOT EXISTS group_project.gold")

# 7. Save DimDate in Gold
df_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("group_project.gold.dim_date")

print("DimDate successfully saved in Gold.")